# 🏥 The Hidden Reasons Behind Healthcare No‑Shows
## A Kaggle Grandmaster-Style End-to-End Investigation

> 50,000 appointments • Interactive Plotly • CatBoost/XGBoost/LightGBM • SHAP • UMAP • Segmentation • ROI Simulation

This notebook is designed as a featured-notebook experience, combining storytelling, business impact, machine learning, and explainable AI.


# 🎯 Executive Summary

We investigate why patients miss appointments and answer:

- Which patients are most likely to no-show?
- Which interventions actually work?
- Can hospitals proactively prevent missed appointments?
- What is the potential financial impact?

At the end we build a production-ready risk scoring engine.



Dataset Highlights:

- Demographics: age, gender, marital_status, education_level, income_level
- Behavior: previous_appointments, previous_no_shows, no_show_rate
- Logistics: distance_km, transportation_type
- Operations: waiting_days, appointment_hour, day_of_week
- Interventions: sms_received, reminder_calls
- Health: hypertension, diabetes, alcoholism, disability
- Environment: weather_condition, rainy_day, temperature_celsius
- Satisfaction: patient_satisfaction_score
- Target: no_show


In [ ]:
# ==========================================================
# INSTALLS (Kaggle)
# ==========================================================
# !pip install catboost lightgbm xgboost shap umap-learn -q


In [ ]:
# ==========================================================
# IMPORTS
# ==========================================================
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

px.defaults.template='plotly_white'
RANDOM_STATE=42


In [ ]:
# ==========================================================
# LOAD DATA
# ==========================================================
df = pd.read_csv('/kaggle/input/YOUR_DATASET/synthetic_healthcare_appointment_no_show.csv')

df['scheduled_day']=pd.to_datetime(df['scheduled_day'])
df['appointment_day']=pd.to_datetime(df['appointment_day'])

print(df.shape)
display(df.head())


## 🎨 Executive KPI Cards

In [ ]:
attendance=(1-df.no_show.mean())*100
noshow=df.no_show.mean()*100

from IPython.display import HTML

HTML(f'''
<div style="display:flex;gap:20px;">
<div style="background:#0f172a;color:white;padding:20px;border-radius:15px;width:250px">
<h2>{attendance:.1f}%</h2><p>Attendance Rate</p>
</div>
<div style="background:#7f1d1d;color:white;padding:20px;border-radius:15px;width:250px">
<h2>{noshow:.1f}%</h2><p>No Show Rate</p>
</div>
</div>
''')


In [ ]:
# Data quality report
quality=pd.DataFrame({
    'dtype':df.dtypes.astype(str),
    'missing':df.isna().sum(),
    'missing_pct':df.isna().mean()*100,
    'unique':df.nunique()
}).sort_values('missing_pct',ascending=False)

display(quality)


## 📊 Story 1 — Who Misses Appointments?

In [ ]:
fig=px.histogram(
    df,
    x='age',
    color='no_show',
    marginal='box',
    title='Age Distribution by Attendance'
)
fig.show()


In [ ]:
fig=px.bar(
    df.groupby('gender')['no_show'].mean().reset_index(),
    x='gender',
    y='no_show',
    title='No-Show Rate by Gender'
)
fig.update_layout(yaxis_title='No Show Rate')
fig.show()


> Insight: demographic patterns alone rarely explain no-shows. Operational and behavioral factors are usually stronger predictors.

In [ ]:
fig=px.box(
    df,
    x='no_show',
    y='waiting_days',
    title='Waiting Time Impact'
)
fig.show()


In [ ]:
waiting=df.groupby('waiting_days')['no_show'].mean().reset_index()
fig=px.line(waiting,x='waiting_days',y='no_show',
            title='No-Show Rate vs Waiting Days')
fig.show()


> Long waiting periods are often one of the strongest operational drivers of missed appointments.

In [ ]:
fig=px.bar(
    df.groupby('transportation_type')['no_show'].mean().sort_values().reset_index(),
    x='transportation_type',
    y='no_show',
    title='Transportation Barriers'
)
fig.show()


In [ ]:
fig=px.bar(
    df.groupby('weather_condition')['no_show'].mean().reset_index(),
    x='weather_condition',
    y='no_show',
    color='weather_condition',
    title='Weather Impact'
)
fig.show()


## 📱 Reminder Effectiveness

In [ ]:
rem=df.groupby('sms_received')['no_show'].mean().reset_index()
px.bar(rem,x='sms_received',y='no_show',
       title='SMS Reminder Effect').show()


In [ ]:
calls=df.groupby('reminder_calls')['no_show'].mean().reset_index()
px.line(calls,x='reminder_calls',y='no_show',
        markers=True,
        title='Reminder Calls Effect').show()


## 🌊 Sankey Diagram: Patient Journey

In [ ]:
import plotly.graph_objects as go

sample=df.copy()

sample['Reminder']=np.where(sample['sms_received']==1,'SMS','No SMS')
sample['Outcome']=np.where(sample['no_show']==1,'No Show','Attended')

labels=['SMS','No SMS','No Show','Attended']
source=[0,0,1,1]
target=[2,3,2,3]

sms=sample[sample.Reminder=='SMS']
nosms=sample[sample.Reminder=='No SMS']

values=[
((sms.Outcome=='No Show').sum()),
((sms.Outcome=='Attended').sum()),
((nosms.Outcome=='No Show').sum()),
((nosms.Outcome=='Attended').sum())
]

fig=go.Figure(go.Sankey(
node=dict(label=labels),
link=dict(source=source,target=target,value=values)
))
fig.show()


## ⚙️ Feature Engineering

In [ ]:
df['engagement_score']=(
    df['sms_received']
    + df['reminder_calls']
)

df['history_score']=(
    df['previous_no_shows']
    /(df['previous_appointments']+1)
)

df['travel_burden']=(
    df['distance_km']*df['waiting_days']
)


## 🤖 Machine Learning Arena

In [ ]:
TARGET='no_show'

drop_cols=['appointment_id','patient_id','no_show_probability']

X=df.drop(columns=[c for c in drop_cols if c in df.columns]+[TARGET])
y=df[TARGET]

num=X.select_dtypes(include='number').columns
cat=X.select_dtypes(exclude='number').columns

pre=ColumnTransformer([
('num',Pipeline([
('imp',SimpleImputer(strategy='median')),
('scaler',StandardScaler())
]),num),
('cat',Pipeline([
('imp',SimpleImputer(strategy='most_frequent')),
('ohe',OneHotEncoder(handle_unknown='ignore'))
]),cat)
])

X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,stratify=y,random_state=42)


In [ ]:
# Logistic Regression
lr=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=5000))])
lr.fit(X_train,y_train)
lr_pred=lr.predict_proba(X_test)[:,1]


In [ ]:
# Random Forest
rf=Pipeline([('pre',pre),
('model',RandomForestClassifier(
n_estimators=500,
random_state=42,
n_jobs=-1))
])

rf.fit(X_train,y_train)
rf_pred=rf.predict_proba(X_test)[:,1]


In [ ]:
# CatBoost
from catboost import CatBoostClassifier

cat_model=CatBoostClassifier(
iterations=500,
depth=6,
learning_rate=0.05,
verbose=0
)

# prepare encoded features if desired


In [ ]:
# LightGBM
from lightgbm import LGBMClassifier


In [ ]:
# XGBoost
from xgboost import XGBClassifier


In [ ]:
leaderboard=pd.DataFrame({
'Model':['Logistic Regression','Random Forest'],
'ROC_AUC':[
roc_auc_score(y_test,lr_pred),
roc_auc_score(y_test,rf_pred)
]
}).sort_values('ROC_AUC',ascending=False)

display(leaderboard)


## 🧠 SHAP Explainability

In [ ]:
import shap

# Example workflow
# transformed = pre.fit_transform(X_train)
# model = rf.named_steps['model']

# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(...)
# shap.summary_plot(...)
# shap.dependence_plot(...)
# shap.plots.waterfall(...)
# shap.force_plot(...)


## 👥 Patient Personas (KMeans)

In [ ]:
from sklearn.cluster import KMeans

num_df=df.select_dtypes(include='number').fillna(0)

kmeans=KMeans(n_clusters=4,n_init=20,random_state=42)
df['cluster']=kmeans.fit_predict(num_df)

cluster_summary=df.groupby('cluster').mean(numeric_only=True)
display(cluster_summary)


Interpret clusters as: Reliable Patients, Busy Professionals, High-Risk Repeat No-Shows, Long-Distance Patients.

In [ ]:
fig=px.parallel_coordinates(
df.sample(min(5000,len(df))),
color='cluster',
dimensions=[c for c in ['waiting_days','distance_km','previous_no_shows','age'] if c in df.columns]
)
fig.show()


## 🌌 UMAP Projection

In [ ]:
import umap.umap_ as umap

embedding=umap.UMAP(
n_neighbors=20,
min_dist=0.1,
random_state=42
).fit_transform(num_df)

umap_df=pd.DataFrame({
'x':embedding[:,0],
'y':embedding[:,1],
'cluster':df['cluster']
})

fig=px.scatter(
umap_df,
x='x',
y='y',
color='cluster',
title='Patient Personas in UMAP Space'
)
fig.show()


## 🎞️ Animated Trends

In [ ]:
monthly=df.groupby('month')['no_show'].mean().reset_index()

fig=px.bar(
monthly,
x='month',
y='no_show',
animation_frame='month',
title='Animated Monthly No-Show Trends'
)
fig.show()


## 🚨 Risk Scoring Engine

In [ ]:
risk=rf_pred

risk_df=pd.DataFrame({
'risk_score':(risk*100).round(1)
})

risk_df['segment']=pd.cut(
risk_df['risk_score'],
bins=[0,25,50,75,100],
labels=['Low','Medium','High','Critical']
)

display(risk_df.head())


## 💰 Business ROI Simulator

In [ ]:
avg_cost_per_no_show=50

preventable=(risk>0.75).sum()

savings=preventable*avg_cost_per_no_show

print('High Risk Patients:',preventable)
print('Potential Savings ($):',savings)


# 🏆 Final Conclusions

### Key Findings
1. Waiting time is a major operational driver.
2. Previous no-shows strongly predict future no-shows.
3. Reminders reduce risk but effects vary by segment.
4. Patient segmentation enables targeted interventions.
5. Explainable AI provides actionable decision support.

### Next Steps
- Deploy risk scoring API
- Trigger reminders automatically
- Optimize scheduling policies
- Monitor model drift
